# Stardew Valley Fishing AI — 8-D Dueling Double DQN on Colab

Train your own fishing AI on 4 parallel environments with GPU acceleration, export it to the official **8-D ONNX contract**, and submit it to the challenge leaderboard!

**Runtime → Change runtime type → T4 GPU** before running.

In [ ]:
# @title 1. Install dependencies & clone repo
import os, sys

BRANCH = "master"  # @param {type:"string"}
REPO = "https://github.com/keethesh/StardewValleyFishingAI.git"

if not os.path.exists("StardewValleyFishingAI"):
    !git clone --branch {BRANCH} {REPO}
%cd StardewValleyFishingAI
!git checkout -f {BRANCH}
!git pull origin {BRANCH} 2>/dev/null

# Install dependencies (including ONNX for model export & verification)
!pip install -q torch numpy matplotlib onnx onnxruntime pygame

# Tell the script we're on Colab
os.environ['COLAB_GPU'] = '1'
print(f"Working in: {os.getcwd()}")

In [ ]:
# @title 2. Verify GPU & 8-D Model Architecture
import torch, sys
sys.path.insert(0, '.')

print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

# Test 8-D environment & model contract
from environment import FishingMinigameEnv, OBS_DIM
e = FishingMinigameEnv(render_mode=None)
state = e.reset()
print(f"Env state dim: {state.shape[0]} (expected {OBS_DIM})")
assert state.shape[0] == 8, f"Expected 8-D state, got {state.shape[0]}"

from main import DQNAgent, VectorizedEnv
v = VectorizedEnv(num_envs=4, render_mode=None)
a = DQNAgent(state_dim=OBS_DIM, action_dim=2)
params = sum(p.numel() for p in a.q_network.parameters())
print(f"Agent params: {params:,} (lean ~13.7k params -> ~56 KB ONNX)")
print("All imports & contract checks PASSED!")

In [ ]:
# @title 3. (Optional) Mount Google Drive for persistent model save
# Checkpoints survive Colab disconnects if saved to Drive
from google.colab import drive
import os

MOUNT_DRIVE = False  # @param {type:"boolean"}

if MOUNT_DRIVE:
    drive.mount('/content/drive')
    DRIVE_PATH = '/content/drive/MyDrive/stardew-fishing-models'
    os.makedirs(DRIVE_PATH, exist_ok=True)
    # Symlink models folder to Drive
    if os.path.islink('models'):
        os.unlink('models')
    !rm -rf models
    !ln -sf "{DRIVE_PATH}" models
    print(f"Models will save to: {DRIVE_PATH}")
else:
    !mkdir -p models/checkpoints
    print("Models save locally (lost on disconnect)")

In [ ]:
# @title 4. Start Training!
# Training uses 4 parallel envs + batched GPU forward pass
# Checkpoints saved every 500 episodes (or at end of run)

NUM_EPISODES = 5000  # @param {type:"integer"}
SAVE_EVERY = 500     # @param {type:"integer"}

os.makedirs("models/checkpoints", exist_ok=True)
os.makedirs("training_logs/graphs", exist_ok=True)
os.makedirs("training_logs/evolution", exist_ok=True)

print(f"Starting: {NUM_EPISODES} episodes on 4 parallel envs (save_every={SAVE_EVERY})")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
print("=" * 60)

!python main.py --episodes {NUM_EPISODES} --save-every {SAVE_EVERY}

In [ ]:
# @title 5. Export to ONNX & Download for the Competition!
from google.colab import files
import glob, os, zipfile

# Find all checkpoints sorted by episode number
ckpts = sorted(
    glob.glob("models/checkpoints/*.pth"),
    key=lambda p: int(os.path.splitext(os.path.basename(p))[0].split("_")[1]) if "_" in p and os.path.splitext(os.path.basename(p))[0].split("_")[1].isdigit() else 0
)

if not ckpts:
    print("No checkpoints found yet! Run training (Cell 4) first.")
else:
    latest_ckpt = ckpts[-1]
    print(f"Latest checkpoint: {latest_ckpt}")

    # Export to official 8-D competition ONNX contract
    output_onnx = "my_model.onnx"
    !python export_onnx.py {latest_ckpt} --output {output_onnx}

    if os.path.exists(output_onnx):
        size_kb = os.path.getsize(output_onnx) / 1024
        print(f"\n Model exported: {output_onnx} ({size_kb:.1f} KB)")
        print("Ready to drag-and-drop into the competition website (/submit or /play)!")
        files.download(output_onnx)

    # Download latest metrics CSV and milestone log
    for pattern, label in [("training_logs/milestones_*.txt", "Milestones"),
                           ("training_logs/training_metrics_*.csv", "Metrics CSV")]:
        matches = sorted(glob.glob(pattern))
        if matches:
            print(f"Downloading {label}: {matches[-1]}")
            files.download(matches[-1])

    # Optional: zip all checkpoints for bulk download
    zip_path = "all_checkpoints.zip"
    with zipfile.ZipFile(zip_path, 'w') as zf:
        for c in ckpts:
            zf.write(c, os.path.basename(c))
    print(f"Created {zip_path} ({len(ckpts)} checkpoints)")
    files.download(zip_path)

In [ ]:
# @title (Optional) Resume / Fine-Tune from uploaded checkpoint
from google.colab import files
import os

uploaded = files.upload()
for fn in uploaded.keys():
    dest = f"models/checkpoints/{fn}"
    os.makedirs("models/checkpoints", exist_ok=True)
    os.rename(fn, dest)
    print(f"Uploaded -> {dest}")

    RESUME_EPISODES = 2000  # @param {type:"integer"}
    print(f"Resuming training for {RESUME_EPISODES} episodes with eps_start=0.2...")
    !python main.py --checkpoint {dest} --episodes {RESUME_EPISODES} --eps-start 0.2

In [ ]:
# @title (Optional) Restore from Google Drive after reconnect
from google.colab import drive
import os

DRIVE_PATH = '/content/drive/MyDrive/stardew-fishing-models'

if os.path.isdir('/content/drive/MyDrive'):
    if os.path.exists(DRIVE_PATH):
        !rm -rf models
        !ln -sf "{DRIVE_PATH}" models
        print(f"Linked models -> {DRIVE_PATH}")
        !ls models/checkpoints/
    else:
        print("No Drive backup found — train from scratch")
else:
    # Mount Drive
    drive.mount('/content/drive')
    if os.path.exists(DRIVE_PATH):
        !rm -rf models
        !ln -sf "{DRIVE_PATH}" models
        print(f"Linked models -> {DRIVE_PATH}")
    else:
        print("No backup found on Drive")